# Module 01: NumPy for Machine Learning
## Notebook 02: Indexing, Slicing, and Reshaping

In machine learning workflows, data rarely arrives in the exact shape or format your algorithms require. You will constantly slice subsets of features, partition samples into train/validation splits, reshape flat data into multi-dimensional images, and manipulate tensor dimensions.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Extract elements, rows, columns, and sub-matrices using multi-dimensional slicing.
2. Distinguish between **memory views** and **independent copies** to avoid subtle data corruption bugs.
3. Reshape arrays using `.reshape()` and understand the `-1` auto-inferred dimension.
4. Add and remove singleton dimensions using `np.newaxis`, `np.expand_dims()`, and `np.squeeze()`.
5. Combine and partition datasets using `np.concatenate()`, `np.vstack()`, and `np.hstack()`.

In [1]:
import numpy as np

print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.3


### 1. 1D Array Indexing and Slicing

Slicing syntax follows the standard Python pattern: `arr[start:stop:step]`
- `start`: Inclusive starting index (defaults to 0)
- `stop`: Exclusive ending index (defaults to length of array)
- `step`: Stride length (defaults to 1; negative step reverses direction)

In [2]:
arr = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])

print("Original array:         ", arr)
print("Single element arr[2]:  ", arr[2])
print("Slice arr[2:7]:         ", arr[2:7])
print("Step slice arr[::2]:    ", arr[::2])
print("Reversed array arr[::-1]:", arr[::-1])
print("Negative indexing arr[-3:]:", arr[-3:])

Original array:          [ 10  20  30  40  50  60  70  80  90 100]
Single element arr[2]:   30
Slice arr[2:7]:          [30 40 50 60 70]
Step slice arr[::2]:     [10 30 50 70 90]
Reversed array arr[::-1]: [100  90  80  70  60  50  40  30  20  10]
Negative indexing arr[-3:]: [ 80  90 100]


---
### 2. Multi-Dimensional Indexing (2D & 3D)

In a 2D array representing a dataset:
- **Axis 0** represents **Rows** (individual samples/observations).
- **Axis 1** represents **Columns** (individual features/variables).

Syntax: `matrix[row_slice, col_slice]`

In [4]:
# Simulated dataset: 5 samples, 4 features
# e.g., [Age, Income, Credit Score, Years Employed]
data = np.array([
    [25, 45000, 710, 3],
    [38, 82000, 680, 8],
    [45, 110000, 750, 15],
    [29, 52000, 620, 2],
    [52, 95000, 800, 20]
])

print("Dataset matrix (5 samples x 4 features):")
print(data)

# Extract a single observation (Row 0)
print("\nFirst sample (Sample 0):", data[0, :])

# Extract a single feature across all samples (Feature 1: Income)
income_feature = data[:, 1]
print("Income column for all samples:", income_feature)

# Extract a sub-matrix (First 3 samples, first 2 features)
sub_matrix = data[:3, :2]
print("\nFirst 3 samples, first 2 features:\n", sub_matrix)

Dataset matrix (5 samples x 4 features):
[[    25  45000    710      3]
 [    38  82000    680      8]
 [    45 110000    750     15]
 [    29  52000    620      2]
 [    52  95000    800     20]]

First sample (Sample 0): [   25 45000   710     3]
Income column for all samples: [ 45000  82000 110000  52000  95000]

First 3 samples, first 2 features:
 [[    25  45000]
 [    38  82000]
 [    45 110000]]


---
### 3. Critical Concept: Memory Views vs. Deep Copies

> **CRITICAL WARNING FOR MACHINE LEARNING:**
> In NumPy, standard array slicing produces a **VIEW** of the existing array, **NOT a copy**!
> If you modify a view, you **silently mutate the original data**.
> To create an independent duplicate, you must explicitly call `.copy()`.

In [5]:
# Demonstrating the 'View' behavior
original_data = np.array([100.0, 200.0, 300.0, 400.0])
view_slice = original_data[:2]

# Modifying the view
view_slice[0] = 999.0

print("view_slice:    ", view_slice)
print("original_data: ", original_data)  # Original was mutated!
print("Does view share memory?", view_slice.base is original_data)

view_slice:     [999. 200.]
original_data:  [999. 200. 300. 400.]
Does view share memory? True


In [6]:
# Safe practice: Explicitly creating a .copy()
safe_original = np.array([100.0, 200.0, 300.0, 400.0])
independent_copy = safe_original[:2].copy()

independent_copy[0] = 999.0

print("\nindependent_copy: ", independent_copy)
print("safe_original:    ", safe_original)  # Original remains intact!
print("Does copy share memory?", independent_copy.base is safe_original)


independent_copy:  [999. 200.]
safe_original:     [100. 200. 300. 400.]
Does copy share memory? False


---
### 4. Reshaping Arrays

Reshaping reorganizes data into a new shape without altering the underlying data elements.
- The total number of elements (`arr.size`) must remain identical!
- The special value `-1` allows NumPy to automatically infer the remaining dimension.

#### Flattening Arrays: `flatten()` vs `ravel()`
- `.flatten()`: Returns a **copy** of the flattened 1D array.
- `.ravel()`: Returns a **view** whenever possible (faster, zero memory overhead).

In [7]:
# Sequence of 12 elements
sequence = np.arange(12)
print("Original 1D array:", sequence)

# Reshape to 3 rows, 4 columns
matrix_3x4 = sequence.reshape(3, 4)
print("\nReshaped to (3, 4):\n", matrix_3x4)

# Using -1 to let NumPy calculate the dimension automatically
# Suppose we want 2 rows and want NumPy to compute columns:
matrix_2x6 = sequence.reshape(2, -1)
print("\nReshaped to (2, -1) -> shape is:", matrix_2x6.shape)

# Flattening: ravel (view) vs flatten (copy)
raveled = matrix_3x4.ravel()
flattened = matrix_3x4.flatten()

print("\nRaveled shape:  ", raveled.shape)
print("Flattened shape:", flattened.shape)
print("Is raveled a view?", raveled.base is not None)
print("Is flattened a view?", flattened.base is not None)

Original 1D array: [ 0  1  2  3  4  5  6  7  8  9 10 11]

Reshaped to (3, 4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Reshaped to (2, -1) -> shape is: (2, 6)

Raveled shape:   (12,)
Flattened shape: (12,)
Is raveled a view? True
Is flattened a view? False


---
### 5. Adding and Removing Singleton Dimensions

Machine learning libraries (like Scikit-Learn or PyTorch) require specific dimensionalities:
- A 1D target vector `y` of shape `(N,)` often needs to be reshaped to a 2D column matrix `(N, 1)`.
- Tools:
  - `np.newaxis` (or `None`)
  - `np.expand_dims(arr, axis)`
  - `np.squeeze(arr)` removes dimensions of length 1.

In [8]:
# 1D Target vector
y = np.array([1.2, 3.4, 5.6, 7.8])
print(f"Original y shape: {y.shape} (ndim: {y.ndim})")

# Method 1: Using np.newaxis / None
y_col = y[:, np.newaxis]
print(f"Using np.newaxis shape: {y_col.shape} (ndim: {y_col.ndim})")

# Method 2: Using np.expand_dims
y_expanded = np.expand_dims(y, axis=0) # Shape: (1, 4) - row vector
print(f"Using np.expand_dims(axis=0) shape: {y_expanded.shape}")

# Squeezing out singleton dimensions
squeezed = np.squeeze(y_col)
print(f"After np.squeeze shape: {squeezed.shape}")

Original y shape: (4,) (ndim: 1)
Using np.newaxis shape: (4, 1) (ndim: 2)
Using np.expand_dims(axis=0) shape: (1, 4)
After np.squeeze shape: (4,)


---
### 6. Stacking and Concatenation

Combining separate feature sets or appending sample batches is a daily ML task.
- `np.concatenate([a, b], axis)`: Joins along an existing axis.
- `np.vstack([a, b])`: Stacks vertically (row-wise, adding samples).
- `np.hstack([a, b])`: Stacks horizontally (column-wise, adding features).
- `np.column_stack([a, b])`: Stacks 1D vectors as columns into a 2D matrix.

In [9]:
# Feature set 1 (e.g., numerical features: Age, Income)
X_num = np.array([
    [25, 50000],
    [32, 65000],
    [47, 90000]
])

# Feature set 2 (e.g., encoded categorical features: Education Level, Job Code)
X_cat = np.array([
    [2, 101],
    [3, 102],
    [1, 101]
])

# Horizontally combine features: shape becomes (3, 4)
X_full = np.hstack([X_num, X_cat])
print("Horizontally stacked feature matrix (3 samples, 4 features):\n", X_full)

# New batch of 2 samples arriving
X_new_batch = np.array([
    [29, 58000, 2, 103],
    [41, 82000, 4, 101]
])

# Vertically combine samples: shape becomes (5, 4)
X_all_samples = np.vstack([X_full, X_new_batch])
print("\nVertically stacked samples (5 samples, 4 features):\n", X_all_samples)

Horizontally stacked feature matrix (3 samples, 4 features):
 [[   25 50000     2   101]
 [   32 65000     3   102]
 [   47 90000     1   101]]

Vertically stacked samples (5 samples, 4 features):
 [[   25 50000     2   101]
 [   32 65000     3   102]
 [   47 90000     1   101]
 [   29 58000     2   103]
 [   41 82000     4   101]]


### Summary & Next Steps
In this notebook, you learned:
- 1D and multi-dimensional slicing mechanics.
- The vital distinction between memory views and `.copy()`.
- Reshaping arrays with auto-inferred dimensions (`-1`).
- Manipulating singleton dimensions using `np.newaxis` and `np.expand_dims`.
- Stacking and concatenating arrays for dataset preprocessing.

**Next Notebook:** `03_vectorization_and_broadcasting.ipynb` — Unlock NumPy's extreme execution speed with element-wise operations and multi-dimensional broadcasting rules.